In [0]:
%pip install psycopg2-binary sqlalchemy

In [0]:
conn = psycopg2.connect(
    host=RDS_HOST,
    database="enem_db",
    user=RDS_USER,
    password=RDS_PASS,
    port=5432,
    connect_timeout=30
)
print("teste de conexao efeutado")
conn.close()

In [0]:
import psycopg2

RDS_HOST = dbutils.secrets.get(scope="aws-credentials", key="rds-host")
RDS_USER = dbutils.secrets.get(scope="aws-credentials", key="rds-user")
RDS_PASS = dbutils.secrets.get(scope="aws-credentials", key="rds-password")
RDS_PORT = 5432
RDS_DB   = "enem_db"

conn = psycopg2.connect(
    host=RDS_HOST, database=RDS_DB,
    user=RDS_USER, password=RDS_PASS,
    port=RDS_PORT, connect_timeout=10
)
print("aqui foi")
conn.close()

In [0]:
conn = psycopg2.connect(
    host=RDS_HOST, database=RDS_DB,
    user=RDS_USER, password=RDS_PASS, port=RDS_PORT
)
cursor = conn.cursor()

sql = """
-- SCHEMAS
CREATE SCHEMA IF NOT EXISTS raw;
CREATE SCHEMA IF NOT EXISTS tru;
CREATE SCHEMA IF NOT EXISTS ref;

-- RAW — dados brutos 1:1 com o INEP

CREATE TABLE IF NOT EXISTS raw.enem_microdados (
    nu_inscricao            VARCHAR(20),
    nu_ano                  VARCHAR(4),
    tp_faixa_etaria         VARCHAR(2),
    tp_sexo                 VARCHAR(1),
    tp_estado_civil         VARCHAR(1),
    tp_cor_raca             VARCHAR(1),
    tp_nacionalidade        VARCHAR(1),
    tp_st_conclusao         VARCHAR(1),
    tp_ano_concluiu         VARCHAR(4),
    tp_escola               VARCHAR(1),
    tp_ensino               VARCHAR(1),
    in_treineiro            VARCHAR(1),
    co_municipio_residencia VARCHAR(10),
    no_municipio_residencia VARCHAR(100),
    co_uf_residencia        VARCHAR(2),
    sg_uf_residencia        VARCHAR(2),
    nu_nota_cn              VARCHAR(10),
    nu_nota_ch              VARCHAR(10),
    nu_nota_lc              VARCHAR(10),
    nu_nota_mt              VARCHAR(10),
    nu_nota_redacao         VARCHAR(10),
    tp_status_redacao       VARCHAR(1),
    q001                    VARCHAR(1),
    q002                    VARCHAR(1),
    q003                    VARCHAR(1),
    q004                    VARCHAR(1),
    q005                    VARCHAR(1),
    q006                    VARCHAR(1),
    q007                    VARCHAR(1),
    q008                    VARCHAR(1),
    q009                    VARCHAR(1),
    q010                    VARCHAR(1),
    dt_carga                TIMESTAMP DEFAULT NOW()
);

-- TRU — dados limpos e tipados

CREATE TABLE IF NOT EXISTS tru.enem_candidatos (
    nu_inscricao            VARCHAR(20) PRIMARY KEY,
    nu_ano                  INTEGER,
    tp_sexo                 VARCHAR(1),
    ds_sexo                 VARCHAR(10),
    tp_cor_raca             INTEGER,
    ds_cor_raca             VARCHAR(20),
    tp_escola               INTEGER,
    ds_escola               VARCHAR(20),
    tp_faixa_etaria         INTEGER,
    ds_faixa_etaria         VARCHAR(30),
    in_treineiro            BOOLEAN,
    co_uf_residencia        INTEGER,
    sg_uf_residencia        VARCHAR(2),
    co_municipio_residencia INTEGER,
    no_municipio_residencia VARCHAR(100),
    nu_nota_cn              NUMERIC(6,2),
    nu_nota_ch              NUMERIC(6,2),
    nu_nota_lc              NUMERIC(6,2),
    nu_nota_mt              NUMERIC(6,2),
    nu_nota_redacao         NUMERIC(6,2),
    nu_media_objetivas      NUMERIC(6,2),
    nu_media_total          NUMERIC(6,2),
    tp_status_redacao       VARCHAR(1),
    q001                    VARCHAR(1),
    q002                    VARCHAR(1),
    q006                    VARCHAR(1),
    dt_carga                TIMESTAMP DEFAULT NOW()
);

CREATE INDEX IF NOT EXISTS idx_tru_uf     ON tru.enem_candidatos(sg_uf_residencia);
CREATE INDEX IF NOT EXISTS idx_tru_ano    ON tru.enem_candidatos(nu_ano);
CREATE INDEX IF NOT EXISTS idx_tru_escola ON tru.enem_candidatos(ds_escola);
CREATE INDEX IF NOT EXISTS idx_tru_media  ON tru.enem_candidatos(nu_media_total DESC);


-- REF — visões analíticas prontas para BI

CREATE TABLE IF NOT EXISTS ref.enem_por_municipio (
    co_municipio            INTEGER,
    no_municipio            VARCHAR(100),
    sg_uf                   VARCHAR(2),
    nu_ano                  INTEGER,
    qt_candidatos           INTEGER,
    qt_treineiros           INTEGER,
    media_cn                NUMERIC(6,2),
    media_ch                NUMERIC(6,2),
    media_lc                NUMERIC(6,2),
    media_mt                NUMERIC(6,2),
    media_redacao           NUMERIC(6,2),
    media_total             NUMERIC(6,2),
    desvio_padrao           NUMERIC(6,2),
    PRIMARY KEY (co_municipio, nu_ano)
);

CREATE TABLE IF NOT EXISTS ref.enem_por_uf (
    sg_uf                   VARCHAR(2),
    nu_ano                  INTEGER,
    qt_candidatos           INTEGER,
    media_cn                NUMERIC(6,2),
    media_ch                NUMERIC(6,2),
    media_lc                NUMERIC(6,2),
    media_mt                NUMERIC(6,2),
    media_redacao           NUMERIC(6,2),
    media_total             NUMERIC(6,2),
    desvio_padrao           NUMERIC(6,2),
    PRIMARY KEY (sg_uf, nu_ano)
);

CREATE TABLE IF NOT EXISTS ref.enem_por_escola (
    ds_escola               VARCHAR(20),
    nu_ano                  INTEGER,
    qt_candidatos           INTEGER,
    media_cn                NUMERIC(6,2),
    media_ch                NUMERIC(6,2),
    media_lc                NUMERIC(6,2),
    media_mt                NUMERIC(6,2),
    media_redacao           NUMERIC(6,2),
    media_total             NUMERIC(6,2),
    PRIMARY KEY (ds_escola, nu_ano)
);

CREATE TABLE IF NOT EXISTS ref.text_clusters (
    nu_inscricao            VARCHAR(20) PRIMARY KEY,
    nu_ano                  INTEGER,
    cluster_id              INTEGER,
    ds_cluster              VARCHAR(50),
    nu_nota_redacao         NUMERIC(6,2),
    sg_uf_residencia        VARCHAR(2),
    ds_escola               VARCHAR(20),
    dt_carga                TIMESTAMP DEFAULT NOW()
);
"""

cursor.execute(sql)
conn.commit()
cursor.close()
conn.close()
print("criados: raw, tru, ref")
print("tabelas criadas")

In [0]:
%sql
ALTER TABLE raw.enem_microdados
    ALTER COLUMN tp_faixa_etaria    TYPE TEXT,
    ALTER COLUMN tp_estado_civil    TYPE TEXT,
    ALTER COLUMN tp_cor_raca        TYPE TEXT,
    ALTER COLUMN tp_nacionalidade   TYPE TEXT,
    ALTER COLUMN tp_st_conclusao    TYPE TEXT,
    ALTER COLUMN tp_ano_concluiu    TYPE TEXT,
    ALTER COLUMN tp_escola          TYPE TEXT,
    ALTER COLUMN tp_ensino          TYPE TEXT,
    ALTER COLUMN in_treineiro       TYPE TEXT,
    ALTER COLUMN tp_status_redacao  TYPE TEXT,
    ALTER COLUMN q001               TYPE TEXT,
    ALTER COLUMN q002               TYPE TEXT,
    ALTER COLUMN q003               TYPE TEXT,
    ALTER COLUMN q004               TYPE TEXT,
    ALTER COLUMN q005               TYPE TEXT,
    ALTER COLUMN q006               TYPE TEXT,
    ALTER COLUMN q007               TYPE TEXT,
    ALTER COLUMN q008               TYPE TEXT,
    ALTER COLUMN q009               TYPE TEXT,
    ALTER COLUMN q010               TYPE TEXT;